Join between candidate and italian_cities
on cap
9/436 = 2% old cap

Join between candidate_cleaned and italian_cities
on cap
9/436 = 2% old cap

Join between candidate_cleaned and italian_cities
on city
1/436 = 0.2% city mismatch

---------

Join between joborder and italian_cities
on cap
1209/35898 = 3.3% old cap

Join between joborder_cleaned and italian_cities
on cap
895/35898 = 2.5% old caps

Join between joborder_cleaned and italian_cities
on city
728/35898 = 2% city mismatch



Ho una tabella candidato CA con info riguardo cap, città, provincia regione. questi campi potrebbero essere sbagliati. ho anche una tabella con tutti i comuni italiani CI e le stesse inso, questi sono sicuramente corretti.

Storicamente in Italia: dato un indirizzo è piu probabile che questo cambia cap o che cambia nome del comune? 
devo decidere se utilizzare CA.cap come lookup in CI e sostituire CA.città con CI.città (se presente), o se utilizzare CA.città come lookup in CI e sostituire CA.cap con CI.cap (se presente)

Soluzione:
i cambi di nome sono molto più rari rispetto a quelli dei cap. Quindi utilizza city (ed eventualmente province come lookup)

I have a candidate table CA which has the following columns:
    - cap
    - city
    - province
    - region
This table may have wrong data. 

I also have a italian_cities table IC. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Regarding the wrong data, these are all the cases:
    - a cap/city is valid (it's avaiable in CA and there is a match in IC)
    - a cap/city is wrong (it's available in CA but there is no match in IC)
    - a cap/city is not available in CA

I want to use IC to fill in the wrong/missing information in CA.

I'll have to work with (cap, city) pairs, because they uniquely identify coordinates that will allow me to compute distance between candidates.
Notice that:
    - a city may have multiple caps. those are always the biggest cities, hence most candidates will be located there.
    - a cap may be assigned to multiple cities. This happens for small village, therefore only a small population of candidates will be located there.

Even when a cap/city is valid, I don't know whether it's correctly assigned to a candidate or not. So I would like to choose one as ground truth and correct the other. Considering the statistics above, I'll chooose cap as ground truth. Is it a correct choise or I'd rather choose city?

For cap I would like to use the following logic:
    - if CA.cap is valid keep it.
    - otherwise, if the CA.city is valid take the corresponding IC.CAP if IC.CAP is unique
    - otherwise null

For city, instead, I would like to use the following logic:
    - if CA.cap is valid, take all the IC.city with that cap: if CA.city belongs to one of those then keep CA.city
    - otherwise if there is only one IC.city associated to CA.cap, take IC.city
    - otherwise null

I would probably keep the old cap, city and store the resolved cap/city in new columns

Does this logic make sense?

------------------------

So far the IC table contain a row for each valid (cap, city) pair.
But while some candidates have both valid cap and city, others may only one of them

Because of this, I would add new rows in the italian_cities table:
    1. (Null, city) one row for each city that has multiple caps: that city will have (lat, lng) equal to the average (lat, lng) for eacy pair (?, CA.city)
    2. (cap, Null) a row for each cap that is shared among multiple cities: that cap will have (lat, lng) equal to the average (lat, lng) for eacy pair (CA.cap, ?)
and also add a location_type which will be either EXACT, CAP_ONLY, CITY_ONLY.

Case 1. will be useful to fetch (lat, lng) of a candidate with invalid cap and valid city, while that city does not have a unique cap
Case 2. will be useful to fetch (lat, lng) of candidate with valid cap and invalid city, while that cap is assigned to multiple cities


------------------------

Then I would create a new table ICM with a row for each pair of (cap, city) in IC, with the precomputed distance in km between them. It will have colums:
    - from_cap
    - from_city
    - to_cap
    - to_city
    - distance

------------------------

Finally I would create a new table CAM with a row for each pair of candidates that are within X chilometers nearby.
I will use ICM to get the distance between candidates.




I have a italian_cities table IC in Snowflake. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Notice that:
    - a city may have multiple caps. so there will be multiple rows with the same city and different caps. those are always the biggest cities, hence most candidates will be located there. 
    - a cap may be assigned to multiple cities. so there will be multiple rows with same cap and different city. This happens for small village, therefore only a small population of candidates will be located there.

I would add new rows in the italian_cities table:
    1. row for each city that has multiple caps, add a row with (Null, city): that row will have (lat, lng) equal to the average (lat, lng) for eacy pair (?, CA.city) in the dataset
    2. row for each cap that is shared among multiple cities, add a row (cap, Null): that row will have (lat, lng) equal to the average (lat, lng) for eacy pair (CA.cap, ?)

For the other columns, take them only if all rows have the same value. For example, if the region of all cities corresponding to the same cap are the same, take the region, otherwise set it to null.

and also add a location_type column which will be either EXACT, CAP_ONLY, CITY_ONLY.

THIS IS DONE

---------------------------------------------------------------------------------------


I have a candidate table CA which has the following columns:
    - cap
    - city
    - province
    - region
This table may have wrong data. 

I also have a italian_cities table IC. This table has a row for each pair (cap, city), and the following columns:
    - cap
    - city
    - province
    - region
    - lat
    - lng

Regarding the wrong data, these are all the cases:
    - a cap/city is valid (it's avaiable in CA and there is a match in IC)
    - a cap/city is wrong (it's available in CA but there is no match in IC)
    - a cap/city is not available in CA

I want to use IC to fill in the wrong/missing information in CA.

I'll have to work with (cap, city) pairs, because they uniquely identify coordinates that will allow me to compute distance between candidates.
Notice that:
    - a city may have multiple caps. those are always the biggest cities, hence most candidates will be located there.
    - a cap may be assigned to multiple cities. This happens for small village, therefore only a small population of candidates will be located there.

Even when a cap/city is valid, I don't know whether it's correctly assigned to a candidate or not. So I would like to choose one as ground truth and correct the other. Considering the statistics above, I'll chooose cap as ground truth.

For cap_resolved I would like to use the following logic:
    - if CA.cap is valid keep it.
    - otherwise, if the CA.city is valid take the corresponding IC.CAP if IC.CAP is unique
    - otherwise null

For city_resolved, instead, I would like to use the following logic:
    - if CA.cap_resolved is valid, take all the IC.city with that cap: if CA.city belongs to one of those then keep CA.city
    - otherwise if there is only one IC.city associated to CA.cap_resolved, take IC.city
    - otherwise if CA.cap_resolved is null, and CA.city is a valid city, keep it


I would probably keep the old cap, city and store the resolved cap/city in new columns


CAP	    CITY	        RESOLVED_CAP	RESOLVED_CITY
40012	bologna	        40012	        calderara di reno   -> 40012 is associated to Calderara di Reno, not Bologna
39011	lana lana	    39011	        lana                -> 39011 is associated to Lana, not Lana Lana (which does not exist)
40056	valsamoggia	    40053	        valsamoggia         -> 40056 does not exist, instead 40056 is the cap associated (uniquely) to Valsamoggia
57100	livorno		    Null            livorno             -> 57100 is an old generic cap for Livorno (which now has multiple caps)
80020	napoli	        80020           Null                -> 80020 is associated to a few small villages in the Napoli province, not Napoli
46020	borgo mantovano	46020	        NUll                -> 46020 is associated to a few small villages, not Borgo Mantovano (which is a placeholder for a generic village)
---------------------------------------------------------------------------------------




Once this is done, get the coordinates from IC table for the candidates based on CA.cap_resolved and CA.city_resolved (they don't need t be both non null)
For province and region, get them from IC only when all rows corresponding to CA.cap_resolved and CA.city_resolved share the same value:
for example :
(71121, Foggia) -> [(71121, Foggia, Foggia, Puglia)] take province and region
(Null, Foggia) -> [(71121, Foggia, Foggia, Puglia), (71122, Foggia, Foggia, Puglia)] take provnce and region
(Null, Foggia) -> [(71121, Foggia, Foggia, Puglia), (71122, Foggia, Foggia, Basilicata)] take province and not region

CAP     CITY        REGION          CAP_RESOLVED    CITY_RESOLVED   PROVINCE_RESOLVED   REGION_RESOLVED
30100  	mestre	    veneto                                                                                  -> 30100 does not exist anymore and city is wrong, keep original region
41100		        emilia romagna                                                                          -> 41100 does not exist anymore and city is missing, can't determine province and region
12071	bragnasco	piemonte	    12071			                                                        -> Bragnasco does not exist, 12071 is assigned to two villages which belong to different provinces/regions



CONTROLLARE CHE COORDINATE, PROVINCE_RESOLVED E REGION_RESOLVED SONO CORRETTE

FARLO ANCHE PER JOBORDER
RICONTROLLARE PER JOBORDER
---------------------------------------------------------------------------------------



Now I want to create a distance table DT that contains a row for each pair or rows in IC, and the following columns:
    - from_cap
    - from_city
    - to_cap
    - to_city
    - distance (in km)



I have a Snowflake italian_cities table. It contains aroung 8400 rows. a row for each zip code, with the following columns:
    - cap
    - cap_id: unique identifier based on cap. The same CAP always receives the same identifier by ordering rows by CAP
    - province
    - region
    - lat
    - lng

I want to create another table that will have one row for each combination of these zipcodes, and their distance in KM.

I will query this column in order to get the distance between the location of a candidate and the location of an opened job position (they both have their zip code). If it needs creating any additional column or index to make the search quicker, do it.

Implement a solution in Snowpark. 


For he JOBORDER.PARTTIME_PERCENT in snowflake I have a varchar column with values like  
['0' '10' '10%' '10,00' '10,38' '100' '100%' '11,25' '12,5%' '12,50'
 '13,16' '13,75' '15' '15,00' '16,25' '16,88' '17,50' '18,75' '19' '2,5'
 '20' '20,00' '20,78' '22,50' '24' '24 ore' '24,00' '25' '25 ore' '25,00'
 '27,28' '27,50' '28,75' '30' '30 ore' '30,00' '31,57' '32,50' '35,00'
 '35,56' '36,84' '37' '37,5' '37,50' '40' '40,00' '41,67' '42,50' '43,00'
 '44,44' '45' '45%' '45,00' '46,15' '47,36' '48' '50' '50%' '50% 75%'
 '50%-75%' '50,00' '51%' '51,00' '51,28' '51,95' '52,00' '52,5' '52,50'
 '53,33' '54,05' '55,00' '55,27' '55,75' '56,00' '56,25' '60' '60%'
 '60,00' '60,53' '62,00' '62,5' '62,5%' '62,5%-75% del tempo pieno'
 '62,50' '63,15' '63,16' '64%' '64,10' '64,90' '65' '65%' '65,00' '65,78'
 '66' '66,00' '66,6' '66,67' '67,50' '67,56' '69,44' '70,00' '75' '75%'
 '75,00' '76,92' '77,50' '78,25' '78,94' '78,95' '80' '80,00' '81,25' '82'
 '82,5' '82,50' '83,30' '83,33' '83,34' '84,00' '84,21' '85,00' '87,5'
 '87,50' '90,00' '93,50' '93,75' '94,74' '95,00' '95,83' '97,50' 'no'
 'part time' 'si' None]

I want to convert it to float, following these rules:
'40' -> 40
'60,00' -> 60
'10,25' -> 10.25
'10%' -> 10
'10,3%' -> 10.3
'20 ore' -> 50 (20 ore diviso 40 ore è uguale a 0.5, quindi 50)
'24 ore' -> 60 (24 ore diviso 40 ore è uguale a 0.6, quindi 60)
'40-60' -> 60 (prendi il massimo quando trovi due numeri o piu numeri separati da -)
'50% 75%' -> 75 (prendi il massimo quando trovi due o piu numeri separati da spazio)
'part time' -> 50
'si' -> 50
'no' -> 100
'None' -> 100
qualsiasi altra cosa -> 100

